# Creating the text-to-query prompt

With our database tools defined, we need to make the LLM aware of these tools and how to use them. To do this, we will construct a prompt containing a clear instruction about how to use the database tools, and insert the tool names.

**Run the cell below to install the necessary libraries.**

In [6]:
!pip install -q langchain-openai==0.3.28 langgraph==0.6.3 pymongo==4.13.2 langchain-mongodb==0.6.2  pydantic==2.11.9


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip


**Run the hidden cell below to re-define the MongoDB database toolkit and LLM.**

In [7]:
import os
from pymongo import MongoClient
from langchain_mongodb.agent_toolkit.database import MongoDBDatabase
from langchain_mongodb.agent_toolkit.toolkit import MongoDBDatabaseToolkit
from langchain_openai import ChatOpenAI

MONGODB_URI = os.environ["MONGODB_URI"]
mongodb_client = MongoClient(MONGODB_URI)

# Create the database tools
db = MongoDBDatabase.from_connection_string(connection_string=MONGODB_URI, database="sample_mflix")
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)
toolkit = MongoDBDatabaseToolkit(db=db, llm=llm)
tools = toolkit.get_tools()

### Create LLM prompt

In addition to a prebuilt set of tools, the MongoDB database toolkit also provides an LLM system prompt containing instructions on how to generate MongoDB queries from natural language, and also guidance on how to use the tools available in the toolkit.

We will use this as the system prompt for the LLM in our text-to-query agent.

**Import and preview the prebuilt `MONGODB_AGENT_SYSTEM_PROMPT` to get a feel for the instructions.**

In [8]:
from langchain_mongodb.agent_toolkit import MONGODB_AGENT_SYSTEM_PROMPT

# Preview the system prompt
MONGODB_AGENT_SYSTEM_PROMPT

'You are an agent designed to interact with a MongoDB database.\nGiven an input question, create a syntactically correct MongoDB query to run, then look at the results of the query and return the answer.\nUnless the user specifies a specific number of examples they wish to obtain, always limit your query to at most {top_k} results.\nYou can order the results by a relevant field to return the most interesting examples in the database.\nNever query for all the fields from a specific collection, only ask for the relevant fields given the question.\n\nYou have access to tools for interacting with the database.\nOnly use the below tools. Only use the information returned by the below tools to construct your final answer.\nYou MUST double check your query before executing it. If you get an error while executing a query, rewrite the query and try again.\n\nDO NOT make any update, insert, or delete operations.\n\nThe query MUST include the collection name and the contents of the aggregation pi

Notice that this system prompt takes a `top_k` variable to specify the default number of results to return. We will pre-fill this rather than it being user-specified.

In addition to the `MONGODB_AGENT_SYSTEM_PROMPT`, we will provide some more additional system instructions to the LLM. We will also need to create a placeholder to pass user messages, tool outcomes, chat history etc. to the LLM.
Let's do this using the `ChatPromptTemplate` class like we did before.

**Define a prompt template containing `MONGODB_AGENT_SYSTEM_PROMPT` to instruct the text-to-query agent on how to use the tools provided.**

In [9]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Create a templated prompt for the LLM
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", MONGODB_AGENT_SYSTEM_PROMPT),
        ("system", "Do not re-run tools unless absolutely necessary. If you are not able to get enough information using the tools, reply with I DON'T KNOW. You have access to the following tools: {tool_names}."),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

You can partially fill certain variables in prompt templates if you have access to them before others using the `partial` method in LangChain.

Let's set the value of `top_k` in the `MONGODB_AGENT_SYSTEM_PROMPT` to 5 to return the top 5 results from a query by default, and the `tool_names` in the custom system prompt to a comma separated list of the tools present in the MongoDB database toolkit.

In [10]:
# Pre-fill top_k and tool_names in the prompt template
prompt = prompt.partial(top_k=5, tool_names=", ".join([tool.name for tool in tools]))

### Bind tools to the LLM

Next, let's give the `llm` access to the tools. To do this, we will use the `.bind_tools()` method in LangChain.

**Bind the tools with the OpenAI LLM.**

In [11]:
# Bind tools to the LLM
tool_augmented_llm = llm.bind_tools(tools)

**Chain the `prompt` to the `tool_augmented_llm` using the `|` operator.**

In [12]:
# Chain the prompt and tool-augmented LLM
llm_with_tools = prompt | tool_augmented_llm

# Test the llm_with_tools chain
llm_with_tools.invoke({"messages": [("user", "Give me the top 5 directors by IMDB rating, who have made more than 20 movies")]}).tool_calls

[{'name': 'mongodb_list_collections',
  'args': {},
  'id': 'call_r4Pybdjrv4OMSqNMIfYVoueO',
  'type': 'tool_call'}]

The above test shows that the LLM will first call the `mongodb_list_collections` tool, given a user query.

This is exactly what we want the agent to do as the first step in executing a text-to-query workflow.

Next, let's use the tools from before and the tool-augmented LLM to create the nodes of our agent's graph.